## Download and unzip cornell movie dialogs

In [ ]:
def download_and_decompress(url, storage_path, storage_dir):
   import os.path
   directory = storage_path + "/" + storage_dir
   zip_file = directory + ".zip"
   a_file = directory + "/cornell movie-dialogs corpus/README.txt"
   if not os.path.isfile(a_file):
       import urllib.request
       import zipfile
       urllib.request.urlretrieve(url, zip_file)
       with zipfile.ZipFile(zip_file, "r") as zfh:
           zfh.extractall(directory)
   return

In [1]:
import re

In [8]:
storage_path = "/Users/d0k008o/learning3/chatbot"
storage_dir = "cornell_movie-dialogs_corpus"

## Read conversation lists from movie_conversations
### returns list of list

In [29]:
def read_conversations(storage_path, storage_dir):
   filename = storage_path + "/" + storage_dir + "/movie_conversations.txt"
   with open(filename, "r", encoding="ISO-8859-1") as fh:
       conversations_chunks = [line.split(" +++$+++ ") for line in fh]
   return [re.sub('[\[\]\']', '', el[3].strip()).split(", ") for el in conversations_chunks]

In [30]:
conversations_list = read_conversations(storage_path,storage_dir)

## Read movie lines

In [31]:
def read_lines(storage_path, storage_dir):
   filename = storage_path + "/" + storage_dir + "/movie_lines.txt"
   with open(filename, "r", encoding="ISO-8859-1") as fh:
       lines_chunks = [line.split(" +++$+++ ") for line in fh]
   return {line[0]: line[-1].strip() for line in lines_chunks}

In [32]:
lines_dict = read_lines(storage_path, storage_dir)

## Get tokenized sequential sentences for bot

In [33]:
def get_tokenized_sequencial_sentences(list_of_lines, line_text):
   for line in list_of_lines:
       for i in range(len(line) - 1):
           yield (line_text[line[i]].split(" "), line_text[line[i+1]].split(" "))

#### For yield keyword, refer: https://www.geeksforgeeks.org/python-yield-keyword/
yield is a keyword in Python that is used to return from a function without destroying the states of its local variable and when the function is called, the execution starts from the last yield statement. Any function that contains a yield keyword is termed as generator.

## Wrap up data retrieval and initial processing

In [34]:
def retrieve_cornell_corpora(storage_path="/Users/d0k008o/learning3/chatbot", storage_dir="cornell_movie-dialogs_corpus"):
   #download_and_decompress("http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip",storage_path,storage_dir)
   conversations = read_conversations(storage_path, storage_dir)
   lines = read_lines(storage_path, storage_dir)
   return tuple(zip(*list(get_tokenized_sequencial_sentences(conversations, lines))))

#### Retrieve from cornell_corpora set of conversations as Q & A

In [36]:
sen_l1, sen_l2 = retrieve_cornell_corpora()
#cornell_corpora = tuple(zip(*list(get_tokenized_sequencial_sentences(conversations_list, lines_list))))

#### Example

In [39]:
print("# Two consecutive sentences in a conversation")
print("Q:", sen_l1[0])
print("A:", sen_l2[0])
print("# Corpora length (i.e. number of sentences)")
print(len(sen_l1))
## Check if we have equal Q & A
assert len(sen_l1) == len(sen_l2)

# Two consecutive sentences in a conversation
Q: ['Can', 'we', 'make', 'this', 'quick?', '', 'Roxanne', 'Korrine', 'and', 'Andrew', 'Barrett', 'are', 'having', 'an', 'incredibly', 'horrendous', 'public', 'break-', 'up', 'on', 'the', 'quad.', '', 'Again.']
A: ['Well,', 'I', 'thought', "we'd", 'start', 'with', 'pronunciation,', 'if', "that's", 'okay', 'with', 'you.']
# Corpora length (i.e. number of sentences)
221616


In [42]:
from corpora_tools import *

## Standardize the tokens
clean the punctuation in the sentences, lowercase them and limits their size to 20 words maximum

In [43]:
clean_sen_l1 = [clean_sentence(s) for s in sen_l1]
clean_sen_l2 = [clean_sentence(s) for s in sen_l2]
filt_clean_sen_l1, filt_clean_sen_l2 = filter_sentence_length(clean_sen_l1, clean_sen_l2)
print("# Filtered Corpora length (i.e. number of sentences)")
print(len(filt_clean_sen_l1))
assert len(filt_clean_sen_l1) == len(filt_clean_sen_l2)

# Filtered Corpora length (i.e. number of sentences)
140261


#### build two dictionaries of words
then encode all the words in the corpora with their dictionary indexes.
they should look the same (since the same sentence appears once on the left side, and once in the right side) except there might be some changes introduced by the first and last sentences of a conversation (they appear only once).

In [45]:
dict_l1 = create_indexed_dictionary(filt_clean_sen_l1, dict_size=15000, storage_path="/tmp/l1_dict.p")
dict_l2 = create_indexed_dictionary(filt_clean_sen_l2, dict_size=15000, storage_path="/tmp/l2_dict.p")
idx_sentences_l1 = sentences_to_indexes(filt_clean_sen_l1, dict_l1)
idx_sentences_l2 = sentences_to_indexes(filt_clean_sen_l2, dict_l2)
print("# Same sentences as before, with their dictionary ID")
print("Q:", list(zip(filt_clean_sen_l1[0], idx_sentences_l1[0])))
print("A:", list(zip(filt_clean_sen_l2[0], idx_sentences_l2[0])))

[sentences_to_indexes] Did not find 16823 words
[sentences_to_indexes] Did not find 16649 words
# Same sentences as before, with their dictionary ID
Q: [('well', 68), (',', 8), ('i', 9), ('thought', 141), ('we', 23), ("'", 5), ('d', 83), ('start', 370), ('with', 46), ('pronunciation', 3), (',', 8), ('if', 78), ('that', 18), ("'", 5), ('s', 12), ('okay', 92), ('with', 46), ('you', 7), ('.', 4)]
A: [('not', 31), ('the', 10), ('hacking', 7309), ('and', 23), ('gagging', 8761), ('and', 23), ('spitting', 6354), ('part', 437), ('.', 4), ('please', 145), ('.', 4)]


#### add paddings and markings to the sentences

In [48]:
max_length_l1 = extract_max_length(idx_sentences_l1)
max_length_l2 = extract_max_length(idx_sentences_l2)
data_set = prepare_sentences(idx_sentences_l1, idx_sentences_l2, max_length_l1, max_length_l2)
print("# Prepared minibatch with paddings and extra stuff")
print("Q:", data_set[0][0])
print("A:", data_set[0][1])
print("# The sentence pass from X to Y tokens")
print("Q:", len(idx_sentences_l1[0]), "->", len(data_set[0][0]))
print("A:", len(idx_sentences_l2[0]), "->", len(data_set[0][1]))

# Prepared minibatch with paddings and extra stuff
Q: [0, 68, 8, 9, 141, 23, 5, 83, 370, 46, 3, 8, 78, 18, 5, 12, 92, 46, 7, 4]
A: [1, 31, 10, 7309, 23, 8761, 23, 6354, 437, 4, 145, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]
# The sentence pass from X to Y tokens
Q: 19 -> 20
A: 11 -> 22


## Training the chatbot

### Sum-up data retrieval and pre-processing steps

In [ ]:
def build_dataset(use_stored_dictionary=False):
   sen_l1, sen_l2 = retrieve_cornell_corpora()
   clean_sen_l1 = [clean_sentence(s) for s in sen_l1]#[:30000] ### OTHERWISE IT DOES NOT RUN ON MY LAPTOP
   clean_sen_l2 = [clean_sentence(s) for s in sen_l2]#[:30000] ### OTHERWISE IT DOES NOT RUN ON MY LAPTOP
   filt_clean_sen_l1, filt_clean_sen_l2 = filter_sentence_length(clean_sen_l1, clean_sen_l2, max_len=10)
   if not use_stored_dictionary:
       dict_l1 = create_indexed_dictionary(filt_clean_sen_l1, dict_size=10000, storage_path="/tmp/l1_dict.p")
       dict_l2 = create_indexed_dictionary(filt_clean_sen_l2, dict_size=10000, storage_path="/tmp/l2_dict.p")
   else:
       dict_l1 = pickle.load(open(path_l1_dict, "rb"))
       dict_l2 = pickle.load(open(path_l2_dict, "rb"))
   dict_l1_length = len(dict_l1)
   dict_l2_length = len(dict_l2)
   idx_sentences_l1 = sentences_to_indexes(filt_clean_sen_l1, dict_l1)
   idx_sentences_l2 = sentences_to_indexes(filt_clean_sen_l2, dict_l2)
   max_length_l1 = extract_max_length(idx_sentences_l1)
   max_length_l2 = extract_max_length(idx_sentences_l2)
   data_set = prepare_sentences(idx_sentences_l1, idx_sentences_l2, max_length_l1, max_length_l2)
   return (filt_clean_sen_l1, filt_clean_sen_l2), 
           data_set, 
           (max_length_l1, max_length_l2), 
           (dict_l1_length, dict_l2_length)